# Training Performance

Per-epoch training histories from `results/models/training_history_<run>.json` (written by
`src/pipelines/train.py`). All validation metrics are computed on the **scanner-1 H&E validation split**,
i.e. in-domain; out-of-domain behaviour is analysed in `Testing.ipynb`.

**Selected checkpoint.** Every figure marks the epoch whose weights were saved as the best checkpoint and
are therefore the ones evaluated on the test sets — not the epoch with the highest validation AUC. The rule
mirrors `train.py`: runs with a `metadata.monitor_metric` (graph models: `slide_f2`, maximised, ties → last
epoch, `>=`) and runs without (patch CNN, MIL: minimum validation loss, ties → first epoch, `<`).

**Evaluation levels.** `val_auc` means different things per training mode (graph: region level, slide and
patch logged separately; MIL: slide level; patch CNN: tile level). Levels are drawn with different line
styles and are only comparable within a level; even graph-node patches and CNN tiles are different units.
A like-for-like comparison across families is only possible at slide level on the test sets.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

REPO_ROOT = Path("/home/stud/hweber/master_code")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.plot_style import IEEE_COLORS, apply_ieee_style, figsize

MODELS_DIR = REPO_ROOT / "results/models"
OUT_DIR = REPO_ROOT / "notebooks/results/training"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
# {label: path to training_history_<run>.json} – relative to the repository root or absolute.
# Order defines the colours, markers and line styles (keep it identical to Testing.ipynb).
# Note: train.py overwrites the history file when a run is retrained under the same name.
RUNS = {
    "Graph · GigaPath": (
        "results/models/training_history_e1-graph-plain-noaug_inflammation_dataset-gsdb_cfg-model_config_tissue_region_gsdb_inflammation_augablation_00_none_consistoff_gnn_tissue_region_net_graph_pov-gigapath_heonly_segmentation.json"
    ),
    "MIL · ResNet-18": (
        "results/models/training_history_e1-mil-resnet18_inflammation_dataset-gsdb_cfg-model_config_mil_mil_attention_mil_image_resnet18_heonly.json"
    ),
    "Patch · GigaPath": (
        "results/models/training_history_e1-patch-gigapath_inflammation_dataset-gsdb_cfg-model_config_cnn_gigapath_image_heonly.json"
    ),
    "Patch · ResNet-18": (
        "results/models/training_history_e1-patch-resnet18_inflammation_dataset-gsdb_cfg-model_config_cnn_resnet18_image_heonly.json"
    ),
}

# Level of 'val_auc' for runs that do not log val_slide_/val_region_/val_patch_auc.
# Otherwise inferred from the config name (model_config_mil -> slide, model_config_cnn -> patch).
AUC_LEVEL_OVERRIDE = {}   # e.g. {"MIL · ResNet-18": "slide"}

AUC_LEVELS = ["slide", "region", "patch"]
SMOOTHING = 0.0           # EMA factor in [0, 1) for display only; 0 = raw values
SAVE_PREFIX = "e1"        # None = do not save

## Load

In [ ]:
LEVEL_NAMES = {"slide": "Slide-level", "region": "Region-level", "patch": "Patch/tile-level"}
LEVEL_STYLE = {"slide": "-", "region": "--", "patch": ":"}


def resolve(path):
    p = Path(path)
    p = p if p.is_absolute() else REPO_ROOT / p
    if not p.is_file():
        raise FileNotFoundError(f"Training history not found: {p}")
    return p


def auc_by_level(label, path, hist):
    levels = {lvl: hist[f"val_{lvl}_auc"] for lvl in AUC_LEVELS if hist.get(f"val_{lvl}_auc")}
    if levels:
        return levels
    level = AUC_LEVEL_OVERRIDE.get(label)
    if level is None:
        if "model_config_mil" in path.name:
            level = "slide"
        elif "model_config_cnn" in path.name:
            level = "patch"
        else:
            raise ValueError(f"Unknown level of val_auc for '{label}' – set AUC_LEVEL_OVERRIDE.")
    return {level: hist["val_auc"]} if hist.get("val_auc") else {}


def selected_epoch(hist):
    # 0-based index of the saved best checkpoint, following train.py's comparison rules.
    meta = hist.get("metadata", {})
    metric = meta.get("monitor_metric")
    if metric:
        values = np.asarray(hist.get(f"val_{metric}") or hist[metric], dtype=float)
        mode = meta.get("monitor_mode", "max")
        best = values.max() if mode == "max" else values.min()
        return int(np.flatnonzero(values == best)[-1]), f"val {metric.replace('_', ' ')} ({mode})"
    values = np.asarray(hist["val_loss"], dtype=float)
    return int(np.argmin(values)), "val loss (min)"


runs = {}
for i, (label, history_path) in enumerate(RUNS.items()):
    path = resolve(history_path)
    with open(path) as f:
        hist = json.load(f)
    sel, rule = selected_epoch(hist)
    runs[label] = {"path": path, "hist": hist, "color": IEEE_COLORS[i % len(IEEE_COLORS)],
                   "marker": ["o", "s", "^", "D", "v", "P"][i % 6],   # redundant to colour (colour-blind safe)
                   "auc": auc_by_level(label, path, hist), "sel": sel, "rule": rule,
                   "epochs": len(hist["train_loss"])}
    print(f"{label:18s} {runs[label]['epochs']:3d} epochs  selected epoch {sel + 1:3d}  [{rule}]  "
          f"AUC levels: {', '.join(runs[label]['auc'])}")

## 1 · Training overview

One row per model: validation AUC (left; line style = evaluation level) and training / validation loss
(right; own scale per model). The vertical line marks the selected checkpoint.

In [ ]:
def ema(values, alpha):
    values = np.asarray(values, dtype=float)
    if alpha <= 0:
        return values
    out = np.empty_like(values)
    out[0] = values[0]
    for t in range(1, len(values)):
        out[t] = alpha * out[t - 1] + (1 - alpha) * values[t]
    return out


def save(fig, name):
    if SAVE_PREFIX:
        for ext in ("png", "pdf"):
            fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_{name}.{ext}", bbox_inches="tight")
        print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_{name}.png")


apply_ieee_style()
W, _ = figsize("double")
all_auc = np.concatenate([v for r in runs.values() for v in r["auc"].values()])
sel_auc = [np.asarray(v)[min(r["sel"], len(v) - 1)] for r in runs.values() for v in r["auc"].values()]
y_low = max(0.0, np.floor((min(np.percentile(all_auc, 5), min(sel_auc)) - 2) / 5) * 5)

n = len(runs)
fig, axes = plt.subplots(n, 2, figsize=(W, 1.45 * n + 0.7), gridspec_kw={"width_ratios": [1.25, 1]},
                         squeeze=False)
for row, (label, r) in enumerate(runs.items()):
    ax_auc, ax_loss = axes[row]
    epochs = np.arange(1, r["epochs"] + 1)
    for lvl in AUC_LEVELS:
        if lvl in r["auc"]:
            ax_auc.plot(epochs[:len(r["auc"][lvl])], ema(r["auc"][lvl], SMOOTHING), color=r["color"],
                        linestyle=LEVEL_STYLE[lvl], linewidth=1.4, marker=r["marker"],
                        markevery=max(1, r["epochs"] // 8), markersize=3)
    for split, style, alpha in (("train", "--", 0.55), ("val", "-", 1.0)):
        ax_loss.plot(epochs, ema(r["hist"][f"{split}_loss"], SMOOTHING), color=r["color"], linestyle=style,
                     alpha=alpha, linewidth=1.4, marker=r["marker"], markevery=max(1, r["epochs"] // 8),
                     markersize=3, markerfacecolor="white" if split == "train" else r["color"])
    for ax in (ax_auc, ax_loss):
        ax.axvline(r["sel"] + 1, color="0.25", linestyle="-.", linewidth=0.9)
        ax.set_xlim(0.5, r["epochs"] + 0.5)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax_auc.set_ylim(y_low, 100.5)
    ax_auc.set_ylabel("Val. AUC (%)")
    ax_loss.set_ylabel("Loss")
    ax_auc.set_title(f"{label}  ·  selected epoch {r['sel'] + 1}  ·  {r['rule']}", fontsize=8, loc="left")
    if row == n - 1:
        ax_auc.set_xlabel("Epoch")
        ax_loss.set_xlabel("Epoch")

handles = [Line2D([], [], color="0.3", linestyle=LEVEL_STYLE[l], label=f"{LEVEL_NAMES[l]} AUC") for l in AUC_LEVELS]
handles += [Line2D([], [], color="0.3", linestyle="--", alpha=0.55, marker="o", markerfacecolor="white",
                   markersize=3, label="training loss (hollow markers)"),
            Line2D([], [], color="0.3", linestyle="-", marker="o", markersize=3,
                   label="validation loss (filled markers)"),
            Line2D([], [], color="0.25", linestyle="-.", label="selected checkpoint"),
            Line2D([], [], color="0.3", marker="s", markersize=3.5, linestyle="",
                   label="marker shape: model (see row title)")]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.09, 1, 1))
save(fig, "training_overview")
plt.show()

## 2 · Validation AUC per level

Same curves grouped by evaluation level, so that models can be compared where the level is shared.
Dots mark the selected checkpoint (not the maximum).

In [ ]:
levels = [lvl for lvl in AUC_LEVELS if any(lvl in r["auc"] for r in runs.values())]
fig, axes = plt.subplots(1, len(levels), figsize=(W, 3.3), sharey=True, squeeze=False)
for ax, lvl in zip(axes[0], levels):
    for label, r in runs.items():
        if lvl not in r["auc"]:
            continue
        values = np.asarray(r["auc"][lvl], dtype=float)
        sel = min(r["sel"], len(values) - 1)
        ax.plot(np.arange(1, len(values) + 1), ema(values, SMOOTHING), color=r["color"], linewidth=1.5,
                marker=r["marker"], markevery=max(1, len(values) // 8), markersize=3.5,
                label=f"{label}  ({values[sel]:.1f} @ {sel + 1})")
        ax.plot(sel + 1, values[sel], r["marker"], color=r["color"], markersize=6.5, markeredgecolor="black",
                markeredgewidth=0.8, zorder=5)
    ax.set_title(f"{LEVEL_NAMES[lvl]} AUC")
    ax.set_xlabel("Epoch")
    ax.set_ylim(y_low, 100.5)
    h, l = ax.get_legend_handles_labels()
    h.append(Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="black", markersize=6.5,
                    linestyle="", label="large marker: selected checkpoint"))
    ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=6.5, frameon=False,
              title="AUC @ selected epoch", title_fontsize=6.5)
axes[0, 0].set_ylabel("Validation AUC (%)")
fig.tight_layout()
save(fig, "val_auc_by_level")
plt.show()

## 3 · Summary

In [ ]:
rows = []
for label, r in runs.items():
    h, sel = r["hist"], r["sel"]
    row = {"Model": label, "Selection rule": r["rule"], "Selected epoch": f"{sel + 1} / {r['epochs']}",
           "Train loss": h["train_loss"][sel], "Val loss": h["val_loss"][sel],
           "Val − train loss": h["val_loss"][sel] - h["train_loss"][sel]}
    for lvl, values in r["auc"].items():
        values = np.asarray(values, dtype=float)
        row[f"{LEVEL_NAMES[lvl]} AUC @ selected"] = values[min(sel, len(values) - 1)]
        row[f"{LEVEL_NAMES[lvl]} AUC max (epoch)"] = f"{values.max():.1f} ({int(values.argmax()) + 1})"
    rows.append(row)

summary = pd.DataFrame(rows).set_index("Model")
num_cols = [c for c in summary.columns if summary[c].dtype.kind == "f"]
summary.style.format({c: "{:.3f}" if "loss" in c else "{:.1f}" for c in num_cols}, na_rep="–")

In [ ]:
if SAVE_PREFIX:
    summary.to_csv(OUT_DIR / f"{SAVE_PREFIX}_training_summary.csv")
    print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_training_summary.csv")